# Практическая работа №9. Методический ориентир и эталонные ответы
## Эталонное решение (на примере Варианта №01: `my_user_id = 117`)

---

## 1. Критерии оценивания (единый средний уровень)

| Оценка | Требования к сдаче |
| :--- | :--- |
| **«5» (Отлично)** | Выполнены все 5 шагов с индивидуальным `my_user_id = 100 + XX * 17`, заполнена таблица замеров (время ДО/ПОСЛЕ, тип сканирования, размер индексов), удалены временные таблицы `bench_*`, даны верные ответы на все 4 контрольных вопроса. |
| **«4» (Хорошо)** | Выполнены все эксперименты, но допущены мелкие неточности в интерпретации планов (например, не объяснено исчезновение `Sort` или причина замедления `INSERT`) либо даны неполные ответы на 1–2 вопроса. |
| **«3» (Удовл.)** | Выполнен базовый поиск (`Seq Scan` $\to$ `Index Scan`) и создание индексов, но отсутствует замер `INSERT` или сравнение размеров частичного индекса / ответы на вопросы. |
| **«2» (Неуд.)** | Работа не выполнена, либо неверно указан синтаксис создания индексов / не приведены результаты `EXPLAIN ANALYZE`. |

---
## 2. Полный эталонный SQL-скрипт для Варианта №01 (`my_user_id = 117`)

```sql
-- Шаг 1. Подготовка полигона (200 000 строк)
DROP TABLE IF EXISTS bench_with_idx CASCADE;
DROP TABLE IF EXISTS bench_no_idx CASCADE;
DROP TABLE IF EXISTS orders CASCADE;

CREATE TABLE orders (
    order_id    SERIAL PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    order_date  TIMESTAMP NOT NULL,
    status      VARCHAR(20) NOT NULL,
    amount      NUMERIC(10, 2) NOT NULL
);

INSERT INTO orders (user_id, order_date, status, amount)
SELECT
    (floor(random() * 2000) + 1)::INTEGER,
    NOW() - (random() * 365) * INTERVAL '1 day',
    CASE
        WHEN gs % 20 = 0 THEN 'cancelled'
        WHEN gs % 10 = 1 THEN 'paid'
        ELSE 'delivered'
    END,
    ROUND((random() * 4900 + 100)::NUMERIC, 2)
FROM generate_series(1, 200000) AS gs;

ANALYZE orders;

-- Шаг 2. Точечный поиск (Seq Scan -> Bitmap Index Scan / Index Scan)
EXPLAIN ANALYZE SELECT * FROM orders WHERE user_id = 117;

CREATE INDEX idx_orders_user_id ON orders(user_id);
ANALYZE orders;

EXPLAIN ANALYZE SELECT * FROM orders WHERE user_id = 117;

-- Шаг 3. Сортировка и пагинация (устранение узла Sort)
EXPLAIN ANALYZE
SELECT * FROM orders
WHERE user_id = 117
ORDER BY order_date DESC
LIMIT 5;

CREATE INDEX idx_orders_user_date ON orders(user_id, order_date DESC);
ANALYZE orders;

EXPLAIN ANALYZE
SELECT * FROM orders
WHERE user_id = 117
ORDER BY order_date DESC
LIMIT 5;

-- Шаг 4. Замер скорости INSERT (цена индексов)
CREATE TABLE bench_no_idx AS SELECT * FROM orders WITH NO DATA;
CREATE TABLE bench_with_idx AS SELECT * FROM orders WITH NO DATA;

CREATE INDEX idx_b_user   ON bench_with_idx(user_id);
CREATE INDEX idx_b_date   ON bench_with_idx(order_date);
CREATE INDEX idx_b_status ON bench_with_idx(status);

EXPLAIN ANALYZE INSERT INTO bench_no_idx SELECT * FROM orders LIMIT 50000;
EXPLAIN ANALYZE INSERT INTO bench_with_idx SELECT * FROM orders LIMIT 50000;

DROP TABLE IF EXISTS bench_no_idx;
DROP TABLE IF EXISTS bench_with_idx;

-- Шаг 5. Частичный индекс (Partial Index)
CREATE INDEX idx_orders_cancelled ON orders(user_id) WHERE status = 'cancelled';
ANALYZE orders;

SELECT
    pg_size_pretty(pg_relation_size('idx_orders_user_id'))   AS full_index_size,
    pg_size_pretty(pg_relation_size('idx_orders_cancelled')) AS partial_index_size;

EXPLAIN ANALYZE
SELECT * FROM orders WHERE user_id = 117 AND status = 'cancelled';
```

---
## 3. Ожидаемые ориентиры планов `EXPLAIN ANALYZE` в PostgreSQL

| Эксперимент | Что должно быть в плане ДО | Типичное время ДО | Что должно быть в плане ПОСЛЕ | Типичное время ПОСЛЕ |
| :--- | :--- | :--- | :--- | :--- |
| **Шаг 2 (Поиск по `user_id`)** | `Seq Scan on orders` + `Filter: (user_id = 117)` | 15 – 45 мс | `Bitmap Heap Scan` / `Index Scan using idx_orders_user_id` | 0.08 – 0.5 мс |
| **Шаг 3 (Поиск + `ORDER BY ... LIMIT 5`)** | `Limit -> Sort (top-N heapsort) -> Bitmap Heap Scan` | 0.4 – 1.5 мс | `Limit -> Index Scan using idx_orders_user_date` *(без `Sort`!)* | 0.04 – 0.15 мс |
| **Шаг 4 (Вставка 50 000 строк)** | Вставка в `bench_no_idx` | 40 – 120 мс | Вставка в `bench_with_idx` (3 индекса) | 200 – 550 мс (в 3–5 раз дольше) |
| **Шаг 5 (Размер индекса)** | `idx_orders_user_id`: ~1.4 – 4.3 MB | — | `idx_orders_cancelled`: ~80 – 240 kB | В 10–20 раз компактнее |

*Примечание: абсолютные миллисекунды зависят от нагрузки на CPU и дискового кэша VPS в момент пары, поэтому главное при проверке — правильная смена узлов плана и соотношение «До/После».*

---

## 4. Имитационная проверка распределения данных и ускорения (локальный эталон)

In [1]:
import sqlite3
import time
import random
import pandas as pd

# Проверим ожидаемое распределение статусов и долю строк для Варианта 01 (user_id = 117)
conn_ref = sqlite3.connect(":memory:")
conn_ref.execute("CREATE TABLE orders (order_id INTEGER PRIMARY KEY, user_id INTEGER, order_date TEXT, status TEXT, amount REAL);")

rng = random.Random(2026)
rows = [
    (
        gs,
        rng.randint(1, 2000),
        f"2026-{(gs % 12) + 1:02d}-{(gs % 28) + 1:02d}",
        "cancelled" if gs % 20 == 0 else ("paid" if gs % 10 == 1 else "delivered"),
        round(rng.uniform(100, 5000), 2)
    )
    for gs in range(1, 200001)
]
conn_ref.executemany("INSERT INTO orders VALUES (?, ?, ?, ?, ?)", rows)
conn_ref.commit()

print("Распределение статусов в генераторе (200 000 строк):")
display(pd.read_sql("SELECT status, COUNT(*) AS cnt, ROUND(COUNT(*) * 100.0 / 200000, 1) AS pct FROM orders GROUP BY status ORDER BY cnt DESC;", conn_ref))

# Сравнение локальных планов и времени (пример для user_id = 117)
t0 = time.perf_counter()
cnt_117 = len(conn_ref.execute("SELECT * FROM orders WHERE user_id = 117").fetchall())
ms_no_idx = (time.perf_counter() - t0) * 1000

conn_ref.execute("CREATE INDEX idx_orders_user_id ON orders(user_id);")
t0 = time.perf_counter()
conn_ref.execute("SELECT * FROM orders WHERE user_id = 117").fetchall()
ms_with_idx = (time.perf_counter() - t0) * 1000

print(f"Строк для user_id = 117: {cnt_117}")
print(f"Локальный замер: без индекса = {ms_no_idx:.2f} мс | с индексом = {ms_with_idx:.3f} мс")
conn_ref.close()

Распределение статусов в генераторе (200 000 строк):
Строк для user_id = 117: 93
Локальный замер: без индекса = 6.42 мс | с индексом = 0.154 мс


status,cnt,pct
delivered,170000,85.0
paid,20000,10.0
cancelled,10000,5.0
